### Setup: load the saved representations


In [ ]:
import os
import time

import numpy as np
import pandas as pd

from representations import (
    load_cleaned_corpus, load_tokenized_reviews,
    COUNT_SETTINGS, EMBEDDING_SETTINGS, MODELS_DIR,
)

corpus_df = load_cleaned_corpus()
tokenized_reviews = load_tokenized_reviews(corpus_df)
print("Reviews:", len(corpus_df), "| Tokens:", sum(len(t) for t in tokenized_reviews))

import joblib
from gensim.models import KeyedVectors
from sklearn.feature_extraction.text import CountVectorizer
from representations import MODEL_FILES, build_and_save_all

if not all(os.path.exists(path) for path in MODEL_FILES.values()):
    print("models/ not found - training and saving all representations once...")
    build_and_save_all()


class LoadedModel:
    """Holds loaded word vectors, so the code below can use model.wv like a trained model."""
    def __init__(self, word_vectors):
        self.wv = word_vectors


cbow_model = LoadedModel(KeyedVectors.load(MODEL_FILES["cbow"]))
skipgram_model = LoadedModel(KeyedVectors.load(MODEL_FILES["skipgram"]))
fasttext_model = LoadedModel(KeyedVectors.load(MODEL_FILES["fasttext"]))
tfidf_vectorizer = joblib.load(MODEL_FILES["tfidf"])

# Count-based matrices for the comparison in Step 11
bow_vectorizer = CountVectorizer(**COUNT_SETTINGS)
bow_matrix = bow_vectorizer.fit_transform(corpus_df["cleaned_text"])
tfidf_matrix = tfidf_vectorizer.transform(corpus_df["cleaned_text"])
bow_sparsity = (1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100
tfidf_sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100

print("Loaded CBOW, Skip-gram, fastText vectors and the TF-IDF vectorizer")

## Step 10: Probing the Embeddings (C1–C3)

In [35]:
embedding_models = {
    "CBOW": cbow_model.wv,
    "Skip-gram": skipgram_model.wv,
    "fastText": fasttext_model.wv,
}

### 10.1 Top-5 most similar words (C1)

In [36]:
query_words = ["great", "boring", "actor", "horror", "plot"]

for word in query_words:
    table = pd.DataFrame({
        name: [f"{w} ({s:.2f})" for w, s in vectors.most_similar(word, topn=5)]
        for name, vectors in embedding_models.items()
    }, index=range(1, 6))
    print(f"Query: '{word}'")
    print(table.to_string())
    print()

Query: 'great'
               CBOW         Skip-gram          fastText
1  excellent (0.77)  excellent (0.82)    great.i (0.93)
2  fantastic (0.76)  wonderful (0.81)     'great (0.87)
3  wonderful (0.75)  fantastic (0.77)  excellent (0.80)
4       fine (0.74)   terrific (0.76)   greatest (0.79)
5   terrific (0.72)  brilliant (0.76)  wonderful (0.79)

Query: 'boring'
                   CBOW             Skip-gram              fastText
1           dull (0.83)           dull (0.87)           bore (0.85)
2           bore (0.82)           bore (0.84)           dull (0.83)
3        tedious (0.78)  uninteresting (0.80)  trivialboring (0.82)
4      pointless (0.74)      pointless (0.78)       boringly (0.80)
5  uninteresting (0.71)        tedious (0.75)        tedious (0.80)

Query: 'actor'
                      CBOW         Skip-gram                 fastText
1           actress (0.72)    actress (0.74)            actors (0.95)
2         performer (0.66)  actresses (0.71)  actors/actresses (0.89

### 10.2 Cosine similarity: similar vs unrelated words (C2)

In [37]:
word_pairs = [
    ("good", "great", "expected similar"),
    ("actor", "kitchen", "expected unrelated"),
]

similarity_table = pd.DataFrame({
    name: [round(float(vectors.similarity(a, b)), 3) for a, b, _ in word_pairs]
    for name, vectors in embedding_models.items()
}, index=[f"{a} – {b} ({label})" for a, b, label in word_pairs])
print(similarity_table.to_string())

                                       CBOW  Skip-gram  fastText
good – great (expected similar)       0.678      0.751     0.687
actor – kitchen (expected unrelated) -0.204      0.145    -0.056


### 10.3 Analogy: actor − man + woman ≈ ? (C3)

In [38]:
for name, vectors in embedding_models.items():
    result = vectors.most_similar(positive=["actor", "woman"], negative=["man"], topn=5)
    print(f"{name:10}", [(w, round(s, 3)) for w, s in result])

CBOW       [('actress', 0.697), ('actors/actresses', 0.558), ('performer', 0.534), ('actresses', 0.465), ('cast', 0.45)]
Skip-gram  [('actress', 0.779), ('actresses', 0.777), ('actors/actresses', 0.66), ('performer', 0.643), ('annik', 0.615)]
fastText   [('actor/actress', 0.829), ('actors/actresses', 0.789), ('actress', 0.763), ('actresses', 0.752), ('actors', 0.73)]


## Step 11: Count-based vs Learned Representations (C4)

In [ ]:
# a misspelling that never appears in the corpus (add "e" until it is not in the vocabulary)
test_word = "moviee"
while test_word in bow_vectorizer.vocabulary_:
    test_word += "e"

comparison = pd.DataFrame({
    "Representation": ["Bag of Words", "TF-IDF", "Word2Vec CBOW", "Word2Vec Skip-gram", "fastText"],
    "Dimensions": [bow_matrix.shape[1], tfidf_matrix.shape[1],
                   cbow_model.wv.vector_size, skipgram_model.wv.vector_size, fasttext_model.wv.vector_size],
    "Zero values %": [round(bow_sparsity, 4), round(tfidf_sparsity, 4),
                      round((cbow_model.wv.vectors == 0).mean() * 100, 4),
                      round((skipgram_model.wv.vectors == 0).mean() * 100, 4),
                      round((fasttext_model.wv.vectors == 0).mean() * 100, 4)],
    f"Has a vector for OOV '{test_word}'": [
        test_word in bow_vectorizer.vocabulary_,
        test_word in tfidf_vectorizer.vocabulary_,
        test_word in cbow_model.wv.key_to_index,
        test_word in skipgram_model.wv.key_to_index,
        True,
    ],
    "What it captures": [
        "word counts per review",
        "word importance per review",
        "word meaning from context (dense)",
        "word meaning from context (dense)",
        "word meaning + spelling pieces (dense)",
    ],
})
print(comparison.to_string(index=False))

## Step 12: Save Models and Vectorize Interface (D1, D2)

### 12.1 Save the TF-IDF vectorizer and the embedding models (D1)

In [40]:
from representations import save_representations

saved_files = save_representations(tfidf_vectorizer, cbow_model, skipgram_model, fasttext_model)
for name, path in saved_files.items():
    print(f"{name:9} {path}  ({os.path.getsize(path) / 1024**2:.1f} MB)")

tfidf     D:\NLP PROJ\models\tfidf_vectorizer.joblib  (3.3 MB)
cbow      D:\NLP PROJ\models\word2vec_cbow.kv  (14.8 MB)
skipgram  D:\NLP PROJ\models\word2vec_skipgram.kv  (14.8 MB)
fasttext  D:\NLP PROJ\models\fasttext.kv  (14.8 MB)


### 12.2 Vectorize a raw document (D2)

In [41]:
from vectorize import vectorize_document

raw_document = "This movie was NOT good!<br /><br />The acting was terrible and the plot was boring."

for representation in ["tfidf", "cbow", "skipgram", "fasttext"]:
    vector = vectorize_document(raw_document, representation)
    if representation == "tfidf":
        print(f"{representation:9} shape {vector.shape}, non-zero values: {vector.nnz}")
    else:
        print(f"{representation:9} shape {vector.shape}, first 5 values: {np.round(vector[:5], 3)}")

tfidf     shape (1, 142754), non-zero values: 9
cbow      shape (100,), first 5 values: [-0.574  0.266  1.12  -0.954  0.177]
skipgram  shape (100,), first 5 values: [ 0.15  -0.086  0.026 -0.194  0.1  ]
fasttext  shape (100,), first 5 values: [-0.523 -2.783  0.621  0.081 -0.396]
